In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.integrate import solve_ivp
from IPython.display import HTML

In [ ]:
# --- Physics Parameters ---
m1 = 3.0  # Mass 1
m2 = 1.0  # Mass 2
L1 = 1.0  # Length 1 (focal length)
L2 = 1.0  # Length 2
g = 9.81  # Gravity

# --- Equations of Motion ---
# Using the exact non-linear Lagrangian derivation for a planar double pendulum
def pendulum_dynamics(t, state):
    theta, w_theta, phi, w_phi = state  # Angles and angular velocities
    delta = theta - phi

    # Denominator shared by both angular acceleration equations
    den = 2 * m1 + m2 - m2 * np.cos(2 * delta)

    # Angular acceleration of the top mass (theta)
    dw_theta = (-g * (2 * m1 + m2) * np.sin(theta)
                - m2 * g * np.sin(theta - 2 * phi)
                - 2 * np.sin(delta) * m2 * (w_phi**2 * L2 + w_theta**2 * L1 * np.cos(delta))) / (L1 * den)

    # Angular acceleration of the bottom mass (phi)
    dw_phi = (2 * np.sin(delta) * (w_theta**2 * L1 * (m1 + m2)
              + g * (m1 + m2) * np.cos(theta)
              + w_phi**2 * L2 * m2 * np.cos(delta))) / (L2 * den)

    return [w_theta, dw_theta, w_phi, dw_phi]

# --- Simulation Setup ---
t_span = (0, 10)  # Simulate for 10 seconds
t_eval = np.linspace(t_span[0], t_span[1], 1000)

# We use small angles (e.g., scaling the eigenvector by 0.05 radians)
# so the small-angle approximation holds and the normal modes remain pure.
scale = 0.05

# Initial Conditions [theta, d(theta)/dt, phi, d(phi)/dt]
# Mode 1: Eigenvector (1, 2) - In-phase, slower frequency
init_mode_1 = [scale * 1, 0.0, scale * 2, 0.0]

# Mode 2: Eigenvector (1, -2) - Out-of-phase, faster frequency
init_mode_2 = [scale * 1, 0.0, scale * -2, 0.0]

# Run the numerical integration
sol_mode_1 = solve_ivp(pendulum_dynamics, t_span, init_mode_1, t_eval=t_eval, method='RK45')
sol_mode_2 = solve_ivp(pendulum_dynamics, t_span, init_mode_2, t_eval=t_eval, method='RK45')

# --- Plotting ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Plot Mode 1
ax1.plot(sol_mode_1.t, sol_mode_1.y[0], label=r'$\theta$ (Top mass)', color='blue', lw=2)
ax1.plot(sol_mode_1.t, sol_mode_1.y[2], label=r'$\phi$ (Bottom mass)', color='orange', lw=2, linestyle='--')
ax1.set_title(r'Normal Mode 1: Eigendirection $(\theta, \phi) = (1, 2)$')
ax1.set_ylabel('Angle (radians)')
ax1.grid(True)
ax1.legend()

# Plot Mode 2
ax2.plot(sol_mode_2.t, sol_mode_2.y[0], label=r'$\theta$ (Top mass)', color='blue', lw=2)
ax2.plot(sol_mode_2.t, sol_mode_2.y[2], label=r'$\phi$ (Bottom mass)', color='orange', lw=2, linestyle='--')
ax2.set_title(r'Normal Mode 2: Eigendirection $(\theta, \phi) = (1, -2)$')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Angle (radians)')
ax2.grid(True)
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Simulation Setup ---
t_span = (0, 10)
frames = 300  # Number of frames for the animation
t_eval = np.linspace(t_span[0], t_span[1], frames)
scale = 0.05

# Init Mode 1: (1, 2)
sol_mode_1 = solve_ivp(pendulum_dynamics, t_span, [scale * 1, 0.0, scale * 2, 0.0], t_eval=t_eval)
# Init Mode 2: (1, -2)
sol_mode_2 = solve_ivp(pendulum_dynamics, t_span, [scale * 1, 0.0, scale * -2, 0.0], t_eval=t_eval)

# --- Coordinate Conversion ---
def get_coords(sol):
    theta, phi = sol.y[0], sol.y[2]
    x1 = L1 * np.sin(theta)
    y1 = -L1 * np.cos(theta)
    x2 = x1 + L2 * np.sin(phi)
    y2 = y1 - L2 * np.cos(phi)
    return x1, y1, x2, y2

x1_m1, y1_m1, x2_m1, y2_m1 = get_coords(sol_mode_1)
x1_m2, y1_m2, x2_m2, y2_m2 = get_coords(sol_mode_2)

# --- Animation Setup ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle('Double Pendulum Normal Modes', fontsize=14)

# Set axis limits
for ax in (ax1, ax2):
    ax.set_xlim(-0.5, 0.5) # Zoomed in since we are using small angles
    ax.set_ylim(-2.2, 0.2)
    ax.set_aspect('equal')
    ax.grid(True)

ax1.set_title('Mode 1: (1, 2)')
ax2.set_title('Mode 2: (1, -2)')

# Line objects for the pendulums
line1, = ax1.plot([], [], 'o-', lw=2, markersize=8, color='blue')
line2, = ax2.plot([], [], 'o-', lw=2, markersize=8, color='orange')

def init():
    line1.set_data([], [])
    line2.set_data([], [])
    return line1, line2

def update(frame):
    # Origin to mass 1 to mass 2
    thisx_1 = [0, x1_m1[frame], x2_m1[frame]]
    thisy_1 = [0, y1_m1[frame], y2_m1[frame]]

    thisx_2 = [0, x1_m2[frame], x2_m2[frame]]
    thisy_2 = [0, y1_m2[frame], y2_m2[frame]]

    line1.set_data(thisx_1, thisy_1)
    line2.set_data(thisx_2, thisy_2)
    return line1, line2

# Create the animation
ani = animation.FuncAnimation(fig, update, frames=frames, init_func=init, blit=True, interval=30)

# Prevent the static empty plot from showing up above the video
plt.close()

# Render the video in Colab
HTML(ani.to_jshtml())

In [ ]:
plt.rcParams['animation.embed_limit'] = 100

# --- Simulation Setup ---
t_span = (0, 30) # 30 seconds
frames = 900     # 30 fps
t_eval = np.linspace(t_span[0], t_span[1], frames)

# Initial Conditions
# Pendulum 1 (Blue):
init_state_1 = [np.pi, 3.0, 3 * np.pi / 2, 2.0]

# Pendulum 2 (Red):
init_state_2 = [np.pi, 3.0 + 0.001, 3 * np.pi / 2, 2.0]

# Solve for both
sol1 = solve_ivp(pendulum_dynamics, t_span, init_state_1, t_eval=t_eval, method='RK45')
sol2 = solve_ivp(pendulum_dynamics, t_span, init_state_2, t_eval=t_eval, method='RK45')

# --- Coordinate Conversion Helper ---
def get_coords(sol):
    theta, phi = sol.y[0], sol.y[2]
    x1 = L1 * np.sin(theta)
    y1 = -L1 * np.cos(theta)
    x2 = x1 + L2 * np.sin(phi)
    y2 = y1 - L2 * np.cos(phi)
    return x1, y1, x2, y2

x1_1, y1_1, x2_1, y2_1 = get_coords(sol1)
x1_2, y1_2, x2_2, y2_2 = get_coords(sol2)

# --- Animation Setup ---
fig, ax = plt.subplots(figsize=(8, 8))

ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-2.5, 2.5)
ax.set_aspect('equal')
ax.grid(True, linestyle='--', alpha=0.6)
ax.set_title('The Butterfly Effect in Double Pendulums\n(Initial difference of 0.001 radians/sec)', fontsize=14)

# Lines for the pendulum rods and masses
line1, = ax.plot([], [], 'o-', lw=2, markersize=8, color='blue', alpha=0.8, label='Pendulum 1')
line2, = ax.plot([], [], 'o-', lw=2, markersize=8, color='red', alpha=0.8, label='Pendulum 2')

# Lines for the traces (we will make them a bit shorter so the screen doesn't get completely painted over)
trace1, = ax.plot([], [], '-', lw=1.5, color='blue', alpha=0.4)
trace2, = ax.plot([], [], '-', lw=1.5, color='red', alpha=0.4)

ax.legend(loc="upper right")

def init():
    line1.set_data([], [])
    line2.set_data([], [])
    trace1.set_data([], [])
    trace2.set_data([], [])
    return line1, line2, trace1, trace2

def update(frame):
    # Update pendulum positions
    line1.set_data([0, x1_1[frame], x2_1[frame]], [0, y1_1[frame], y2_1[frame]])
    line2.set_data([0, x1_2[frame], x2_2[frame]], [0, y1_2[frame], y2_2[frame]])

    # Trace the last 50 frames to keep the screen from getting too cluttered,
    # while still showing the divergent paths
    start_trace = max(0, frame - 50)
    trace1.set_data(x2_1[start_trace:frame], y2_1[start_trace:frame])
    trace2.set_data(x2_2[start_trace:frame], y2_2[start_trace:frame])

    return line1, line2, trace1, trace2

# Create the animation
ani = animation.FuncAnimation(fig, update, frames=frames, init_func=init, blit=True, interval=33)

plt.close()

# Render the video in Colab
HTML(ani.to_jshtml())

In [ ]:
# --- Superposition Setup ---
[a,b] = [0.2,0.05] # initial setup
[c,d] = [0.5 * a + 0.25 * b, 0.5 * a - 0.25 * b]

# Mode 1
init_mode_1 = [c * 1, 0.0, c * 2, 0.0]

# 50% Mode 2
init_mode_2 = [d * 1, 0.0, d * -2, 0.0]

# Arbitrary Starting State (0.05, 0.0)
init_super  = [a, 0.0, b, 0.0]

t_span = (0, 10)
t_eval = np.linspace(t_span[0], t_span[1], 1000)

# Simulate all three independently
sol_m1 = solve_ivp(pendulum_dynamics, t_span, init_mode_1, t_eval=t_eval, method='RK45')
sol_m2 = solve_ivp(pendulum_dynamics, t_span, init_mode_2, t_eval=t_eval, method='RK45')
sol_super = solve_ivp(pendulum_dynamics, t_span, init_super, t_eval=t_eval, method='RK45')

# Calculate the linear combination (addition) of the two separate modes
theta_sum = sol_m1.y[0] + sol_m2.y[0]
phi_sum = sol_m1.y[2] + sol_m2.y[2]

# --- Plotting ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Top Mass (Theta)
ax1.plot(sol_super.t, sol_super.y[0], label='Simulation', color='black', lw=5, alpha=0.3)
ax1.plot(sol_super.t, theta_sum, label='Superposition: Mode 1 + Mode 2', color='red', linestyle='--', lw=2)
ax1.set_title(r'Top Mass ($\theta$): Simulating an Arbitrary Small Angle vs. Summing Normal Modes')
ax1.set_ylabel('Angle (rad)')
ax1.grid(True)
ax1.legend()

# Bottom Mass (Phi)
ax2.plot(sol_super.t, sol_super.y[2], label='Simulation', color='black', lw=5, alpha=0.3)
ax2.plot(sol_super.t, phi_sum, label='Superposition: Mode 1 + Mode 2', color='blue', linestyle='--', lw=2)
ax2.set_title(r'Bottom Mass ($\phi$)')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Angle (rad)')
ax2.grid(True)
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
t_span = (0, 30)
frames = 900
t_eval = np.linspace(t_span[0], t_span[1], frames)

# Simulate all three
sol_m1 = solve_ivp(pendulum_dynamics, t_span, init_mode_1, t_eval=t_eval)
sol_m2 = solve_ivp(pendulum_dynamics, t_span, init_mode_2, t_eval=t_eval)
sol_super = solve_ivp(pendulum_dynamics, t_span, init_super, t_eval=t_eval)

# Mathematical Addition of the Angles
theta_sum = sol_m1.y[0] + sol_m2.y[0]
phi_sum = sol_m1.y[2] + sol_m2.y[2]

def get_coords(theta, phi):
    x1 = L1 * np.sin(theta)
    y1 = -L1 * np.cos(theta)
    x2 = x1 + L2 * np.sin(phi)
    y2 = y1 - L2 * np.cos(phi)
    return x1, y1, x2, y2

x1_m1, y1_m1, x2_m1, y2_m1 = get_coords(sol_m1.y[0], sol_m1.y[2])
x1_m2, y1_m2, x2_m2, y2_m2 = get_coords(sol_m2.y[0], sol_m2.y[2])
x1_sup, y1_sup, x2_sup, y2_sup = get_coords(sol_super.y[0], sol_super.y[2])
x1_math, y1_math, x2_math, y2_math = get_coords(theta_sum, phi_sum)

# --- Animation Setup ---
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Superposition Principle: Small-Angle Regime', fontsize=14)

for ax in (ax1, ax2, ax3):
    ax.set_xlim(-0.5, 0.5)
    ax.set_ylim(-2.2, 0.2)
    ax.set_aspect('equal')
    ax.grid(True, linestyle='--', alpha=0.6)

ax1.set_title('Mode 1 Only')
ax2.set_title('Mode 2 Only')
ax3.set_title('Simulated vs. Superposition')

# Pendulum line objects
line_m1, = ax1.plot([], [], 'o-', lw=2, markersize=8, color='blue')
line_m2, = ax2.plot([], [], 'o-', lw=2, markersize=8, color='orange')

# Panel 3 has two pendulums: The simulated one (thick black) and the math sum (red dashed ghost)
line_sup, = ax3.plot([], [], 'o-', lw=4, markersize=10, color='black', alpha=0.4, label='Simulation')
line_math, = ax3.plot([], [], 'o--', lw=2, markersize=6, color='red', label='Superposition')
ax3.legend(loc="upper center")

def init():
    line_m1.set_data([], [])
    line_m2.set_data([], [])
    line_sup.set_data([], [])
    line_math.set_data([], [])
    return line_m1, line_m2, line_sup, line_math

def update(frame):
    line_m1.set_data([0, x1_m1[frame], x2_m1[frame]], [0, y1_m1[frame], y2_m1[frame]])
    line_m2.set_data([0, x1_m2[frame], x2_m2[frame]], [0, y1_m2[frame], y2_m2[frame]])
    line_sup.set_data([0, x1_sup[frame], x2_sup[frame]], [0, y1_sup[frame], y2_sup[frame]])
    line_math.set_data([0, x1_math[frame], x2_math[frame]], [0, y1_math[frame], y2_math[frame]])
    return line_m1, line_m2, line_sup, line_math

ani = animation.FuncAnimation(fig, update, frames=frames, init_func=init, blit=True, interval=33)
plt.close()

HTML(ani.to_jshtml())